In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random
from scipy.optimize import curve_fit
from IPython.display import display
import matplotlib.animation as animation
import matplotlib
matplotlib.use('Agg')
import os
!pip install xlsxwriter

REPEATABLE = True
if REPEATABLE:
    random.seed(42)
    np.random.seed(42)

grid_size = 100
x_vals = np.linspace(0, 10, grid_size)
y_vals = np.linspace(0, 10, grid_size)
X, Y = np.meshgrid(x_vals, y_vals)
num_sensors = 10
timesteps = 10
sensor_range = 2.5
monte_carlo_runs = 50

def generate_sensor(i):
    return {
        'id': f'S{i+1}',
        'pos': [random.uniform(1, 9), random.uniform(1, 9)],
        'range': sensor_range,
        'noise': random.uniform(0.05, 0.2),
        'battery': random.uniform(0.6, 1.0),
        'drift': random.uniform(0.01, 0.1),
        'fail_risk': random.uniform(0.01, 0.2)
    }

def compute_neutrosophic_layers(sensor, X, Y):
    sx, sy = sensor['pos']
    r = sensor['range']
    distance = np.sqrt((X - sx)**2 + (Y - sy)**2)
    coverage = np.clip(1 - distance / r, 0, 1)

    noise = sensor['noise']
    battery = sensor['battery']
    drift = sensor['drift']
    fail_risk = sensor['fail_risk']

    mu = coverage * (1 - noise) * battery
    sigma = (noise + drift + (1 - battery)) * coverage
    nu = ((1 - coverage) * (1 - battery) + fail_risk) * (1 - coverage)
    return mu, sigma, nu, coverage

def neutrosophic_interior(mu, threshold=0.6):
    return mu > threshold

def neutrosophic_exterior(nu, threshold=0.6):
    return nu > threshold

def neutrosophic_boundary(sigma, threshold=0.4):
    return sigma > threshold

def neutrosophic_closure(mu, sigma, threshold_mu=0.4, threshold_sigma=0.2):
    return (mu > threshold_mu) | (sigma > threshold_sigma)

def neutrosophic_subspace(mask):
    return np.where(mask)

sensors = [generate_sensor(i) for i in range(num_sensors)]
results = []

for t in range(timesteps):
    mu_total = np.zeros_like(X)
    sigma_total = np.zeros_like(X)
    nu_total = np.zeros_like(X)
    trust_scores = []

    for sensor in sensors:
        mu_layer, sigma_layer, nu_layer, coverage = compute_neutrosophic_layers(sensor, X, Y)
        mu_total += mu_layer
        sigma_total += sigma_layer
        nu_total += nu_layer

        mask = coverage > 0
        if np.any(mask):
            mu_vals = mu_layer[mask]
            sigma_vals = sigma_layer[mask]
            nu_vals = nu_layer[mask]
            denom = mu_vals + sigma_vals + nu_vals
            norm_trust_vals = np.divide(mu_vals, denom, out=np.zeros_like(mu_vals), where=denom != 0)
            trust_score = np.mean(norm_trust_vals)
        else:
            trust_score = 0

        trust_scores.append({
            'Sensor': sensor['id'],
            'Trust Score': round(trust_score, 4),
            'Battery': round(sensor['battery'], 2),
            'Noise': round(sensor['noise'], 2),
            'Drift': round(sensor['drift'], 2),
            'Fail Risk': round(sensor['fail_risk'], 2)
        })

        sensor['battery'] = max(sensor['battery'] - 0.05, 0)
        sensor['drift'] += 0.005
        sensor['fail_risk'] = min(sensor['fail_risk'] + 0.01, 1.0)

    trust_df = pd.DataFrame(trust_scores).sort_values(by='Trust Score', ascending=False)
    results.append((t, trust_df))

    print(f"\n=== Time Step {t} ===")
    display(trust_df)

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    axs[0].imshow(mu_total, extent=(0, 10, 0, 10), origin='lower', cmap='YlGnBu')
    axs[0].set_title(f"μ - Membership (t={t})")
    axs[1].imshow(sigma_total, extent=(0, 10, 0, 10), origin='lower', cmap='YlOrBr')
    axs[1].set_title(f"σ - Indeterminacy (t={t})")
    axs[2].imshow(nu_total, extent=(0, 10, 0, 10), origin='lower', cmap='Purples')
    axs[2].set_title(f"ν - Non-membership (t={t})")
    for ax in axs:
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
    plt.tight_layout()
    plt.show()

all_runs_scores = []
ranking_frequency = {f"S{i+1}": 0 for i in range(num_sensors)}
final_trust_collection = {f"S{i+1}": [] for i in range(num_sensors)}

for run in range(monte_carlo_runs):
    sensors = [generate_sensor(i) for i in range(num_sensors)]
    sensor_trust_timeline = {sensor['id']: [] for sensor in sensors}

    for t in range(timesteps):
        for sensor in sensors:
            mu_layer, sigma_layer, nu_layer, coverage = compute_neutrosophic_layers(sensor, X, Y)
            mask = coverage > 0
            if np.any(mask):
                mu_vals = mu_layer[mask]
                sigma_vals = sigma_layer[mask]
                nu_vals = nu_layer[mask]
                denom = mu_vals + sigma_vals + nu_vals
                norm_trust_vals = np.divide(mu_vals, denom, out=np.zeros_like(mu_vals), where=denom != 0)
                trust_score = np.mean(norm_trust_vals)
            else:
                trust_score = 0

            sensor_trust_timeline[sensor['id']].append(trust_score)

            sensor['battery'] = max(sensor['battery'] - 0.05, 0)
            sensor['drift'] += 0.005
            sensor['fail_risk'] = min(sensor['fail_risk'] + 0.01, 1.0)

        if t == timesteps - 1:
            final_scores = [(sensor_id, sensor_trust_timeline[sensor_id][-1]) for sensor_id in sensor_trust_timeline]
            for sid, score in final_scores:
                final_trust_collection[sid].append(score)
            sorted_scores = sorted(final_scores, key=lambda x: x[1], reverse=True)
            for rank, (sensor_id, _) in enumerate(sorted_scores[:3]):
                ranking_frequency[sensor_id] += 1

    run_df = pd.DataFrame(sensor_trust_timeline)
    run_df['Timestep'] = list(range(timesteps))
    run_df['Run'] = run
    all_runs_scores.append(run_df)

combined_df = pd.concat(all_runs_scores)
mean_trust = combined_df.groupby('Timestep').mean()
std_trust = combined_df.groupby('Timestep').std()

plt.figure(figsize=(12, 6))
for sensor_id in mean_trust.columns:
    plt.plot(mean_trust.index, mean_trust[sensor_id], label=sensor_id)
    plt.fill_between(mean_trust.index,
                     mean_trust[sensor_id] - std_trust[sensor_id],
                     mean_trust[sensor_id] + std_trust[sensor_id],
                     alpha=0.2)
plt.title("Average Trust Score Over Time with Confidence Bands")
plt.xlabel("Timestep")
plt.ylabel("Normalized Trust Score")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

ranking_df = pd.DataFrame.from_dict(ranking_frequency, orient='index', columns=['Top-3 Appearances'])
ranking_df['Top-3 %'] = (ranking_df['Top-3 Appearances'] / monte_carlo_runs) * 100
ranking_df = ranking_df.sort_values(by='Top-3 %', ascending=False)

summary_stats = []
for sid, scores in final_trust_collection.items():
    summary_stats.append({
        "Sensor": sid,
        "Avg. Final Trust Score": np.mean(scores),
        "Trust StdDev": np.std(scores)
    })
summary_df = pd.DataFrame(summary_stats).sort_values(by="Avg. Final Trust Score", ascending=False)

print("\n📊 Ranking Frequency (% Top-3 appearances):")
display(ranking_df)
print("\n📉 Final Trust Score Summary (Average and Std Dev):")
display(summary_df)

mean_trust.to_csv("montecarlo_mean_trust.csv")
ranking_df.to_csv("montecarlo_ranking_frequency.csv")
summary_df.to_csv("montecarlo_final_stats.csv")

with pd.ExcelWriter("montecarlo_summary.xlsx", engine='xlsxwriter') as writer:
    mean_trust.to_excel(writer, sheet_name='Mean Trust')
    std_trust.to_excel(writer, sheet_name='Trust StdDev')
    ranking_df.to_excel(writer, sheet_name='Ranking Frequency')
    summary_df.to_excel(writer, sheet_name='Final Trust Stats')

print("\n✅ Exported:")
print(" - montecarlo_mean_trust.csv")
print(" - montecarlo_ranking_frequency.csv")
print(" - montecarlo_final_stats.csv")
print(" - montecarlo_summary.xlsx")

sensors_anim = [generate_sensor(i) for i in range(num_sensors)]
mu_frames, sigma_frames, nu_frames = [], [], []

for t in range(timesteps):
    mu_total = np.zeros_like(X)
    sigma_total = np.zeros_like(X)
    nu_total = np.zeros_like(X)

    for sensor in sensors_anim:
        mu_layer, sigma_layer, nu_layer, _ = compute_neutrosophic_layers(sensor, X, Y)
        mu_total += mu_layer
        sigma_total += sigma_layer
        nu_total += nu_layer

        sensor['battery'] = max(sensor['battery'] - 0.05, 0)
        sensor['drift'] += 0.005
        sensor['fail_risk'] = min(sensor['fail_risk'] + 0.01, 1.0)

    mu_frames.append(mu_total.copy())
    sigma_frames.append(sigma_total.copy())
    nu_frames.append(nu_total.copy())

    interior = neutrosophic_interior(mu_total)
    boundary = neutrosophic_boundary(sigma_total)
    exterior = neutrosophic_exterior(nu_total)

os.makedirs("animations", exist_ok=True)

def create_animation(frames, title, filename, cmap):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(frames[0], extent=(0, 10, 0, 10), origin='lower', cmap=cmap)
    ax.set_title(f"{title} (t=0)")
    def update(frame):
        im.set_array(frames[frame])
        ax.set_title(f"{title} (t={frame})")
        return [im]
    ani = animation.FuncAnimation(fig, update, frames=len(frames), blit=True)
    ani.save(f"animations/{filename}.gif", writer='pillow', fps=2)
    plt.close(fig)

create_animation(mu_frames, "Membership (μ)", "mu_animation", 'YlGnBu')
create_animation(sigma_frames, "Indeterminacy (σ)", "sigma_animation", 'YlOrBr')
create_animation(nu_frames, "Non-membership (ν)", "nu_animation", 'Purples')

print("\n✅ Animated heatmaps saved to animations/*.gif")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 3.9 MB/s eta 0:00:00

=== Time Step 0 ===


,Sensor,Trust Score,Battery,Noise,Drift,Fail Risk
5,S6,0.5784,0.99,0.13,0.04,0.11
9,S10,0.5265,0.89,0.08,0.02,0.08
3,S4,0.5012,0.88,0.17,0.04,0.04
6,S7,0.4251,0.83,0.18,0.07,0.02
2,S3,0.4142,0.82,0.15,0.03,0.12
7,S8,0.3571,0.69,0.06,0.02,0.06
0,S1,0.3269,0.69,0.09,0.08,0.14
4,S5,0.3066,0.64,0.06,0.09,0.12
1,S2,0.2943,0.61,0.11,0.03,0.11
8,S9,0.2913,0.68,0.11,0.03,0.19



=== Time Step 1 ===


,Sensor,Trust Score,Battery,Noise,Drift,Fail Risk
5,S6,0.5058,0.94,0.13,0.05,0.12
9,S10,0.4642,0.84,0.08,0.03,0.09
3,S4,0.4408,0.83,0.17,0.05,0.05
6,S7,0.3718,0.78,0.18,0.08,0.03
2,S3,0.3696,0.77,0.15,0.03,0.13
7,S8,0.3154,0.64,0.06,0.02,0.07
0,S1,0.2914,0.64,0.09,0.08,0.15
4,S5,0.2721,0.59,0.06,0.09,0.13
1,S2,0.2597,0.56,0.11,0.03,0.12
8,S9,0.2591,0.63,0.11,0.04,0.20



=== Time Step 2 ===


...


📊 Ranking Frequency (% Top-3 appearances):


,Top-3 Appearances,Top-3 %
S3,24,48.0
S2,19,38.0
S5,17,34.0
S6,15,30.0
S1,14,28.0
S7,14,28.0
S8,14,28.0
S4,12,24.0
S9,11,22.0
S10,10,20.0



📉 Final Trust Score Summary (Average and Std Dev):


,Sensor,Avg. Final Trust Score,Trust StdDev
2,S3,0.143435,0.052067
1,S2,0.135055,0.048434
4,S5,0.132541,0.046072
7,S8,0.129321,0.047537
6,S7,0.128521,0.042124
8,S9,0.126390,0.041497
9,S10,0.125434,0.049970
0,S1,0.124539,0.045995
3,S4,0.124116,0.052434
5,S6,0.118305,0.044766



✅ Exported:
 - montecarlo_mean_trust.csv
 - montecarlo_ranking_frequency.csv
 - montecarlo_final_stats.csv
 - montecarlo_summary.xlsx

✅ Animated heatmaps saved to animations/*.gif


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random
from scipy.optimize import curve_fit
from IPython.display import display
import matplotlib.animation as animation
import matplotlib
matplotlib.use('Agg')
import os
!pip install xlsxwriter

REPEATABLE = True
if REPEATABLE:
    random.seed(42)
    np.random.seed(42)

grid_size = 100
x_vals = np.linspace(0, 10, grid_size)
y_vals = np.linspace(0, 10, grid_size)
X, Y = np.meshgrid(x_vals, y_vals)

num_sensors = 10
timesteps = 10
sensor_range = 2.5
monte_carlo_runs = 50

def generate_sensor(i):
    return {
        'id': f'S{i+1}',
        'pos': [random.uniform(1, 9), random.uniform(1, 9)],
        'range': sensor_range,
        'noise': random.uniform(0.05, 0.2),
        'battery': random.uniform(0.6, 1.0),
        'drift': random.uniform(0.01, 0.1),
        'fail_risk': random.uniform(0.01, 0.2)
    }

def compute_neutrosophic_layers(sensor, X, Y):
    sx, sy = sensor['pos']
    r = sensor['range']
    distance = np.sqrt((X - sx)**2 + (Y - sy)**2)
    coverage = np.clip(1 - distance / r, 0, 1)

    noise = sensor['noise']
    battery = sensor['battery']
    drift = sensor['drift']
    fail_risk = sensor['fail_risk']

    mu = coverage * (1 - noise) * battery
    sigma = (noise + drift + (1 - battery)) * coverage
    nu = ((1 - coverage) * (1 - battery) + fail_risk) * (1 - coverage)
    return mu, sigma, nu, coverage

def neutrosophic_interior(mu, threshold=0.6):
    return mu > threshold

def neutrosophic_exterior(nu, threshold=0.6):
    return nu > threshold

def neutrosophic_boundary(sigma, threshold=0.4):
    return sigma > threshold

def neutrosophic_closure(mu, sigma, threshold_mu=0.4, threshold_sigma=0.2):
    return (mu > threshold_mu) | (sigma > threshold_sigma)

def neutrosophic_subspace(mask):
    return np.where(mask)

sensors_anim = [generate_sensor(i) for i in range(num_sensors)]
mu_frames, sigma_frames, nu_frames = [], [], []
interior_areas, boundary_areas, exterior_areas = [], [], []

for t in range(timesteps):
    mu_total = np.zeros_like(X)
    sigma_total = np.zeros_like(X)
    nu_total = np.zeros_like(X)

    for sensor in sensors_anim:
        mu_layer, sigma_layer, nu_layer, _ = compute_neutrosophic_layers(sensor, X, Y)
        mu_total += mu_layer
        sigma_total += sigma_layer
        nu_total += nu_layer

        sensor['battery'] = max(sensor['battery'] - 0.05, 0)
        sensor['drift'] += 0.005
        sensor['fail_risk'] = min(sensor['fail_risk'] + 0.01, 1.0)

    mu_frames.append(mu_total.copy())
    sigma_frames.append(sigma_total.copy())
    nu_frames.append(nu_total.copy())

    interior = neutrosophic_interior(mu_total)
    boundary = neutrosophic_boundary(sigma_total)
    exterior = neutrosophic_exterior(nu_total)

    interior_areas.append(np.sum(interior))
    boundary_areas.append(np.sum(boundary))
    exterior_areas.append(np.sum(exterior))

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].imshow(interior, extent=(0, 10, 0, 10), origin='lower', cmap='Greens')
    axs[0].set_title(f"Interior Mask (t={t})")

    axs[1].imshow(boundary, extent=(0, 10, 0, 10), origin='lower', cmap='Oranges')
    axs[1].set_title(f"Boundary Mask (t={t})")

    axs[2].imshow(exterior, extent=(0, 10, 0, 10), origin='lower', cmap='Reds')
    axs[2].set_title(f"Exterior Mask (t={t})")

    for ax in axs:
        ax.set_xlabel("X")
        ax.set_ylabel("Y")

    plt.tight_layout()
    plt.savefig(f"animations/neutrosophic_masks_t{t}.png")
    plt.close()

coverage_df = pd.DataFrame({
    'Timestep': list(range(timesteps)),
    'Interior Area': interior_areas,
    'Boundary Area': boundary_areas,
    'Exterior Area': exterior_areas
})

coverage_df.to_csv("neutrosophic_region_areas.csv", index=False)

os.makedirs("animations", exist_ok=True)

def create_animation(frames, title, filename, cmap):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(frames[0], extent=(0, 10, 0, 10), origin='lower', cmap=cmap)
    ax.set_title(f"{title} (t=0)")

    def update(frame):
        im.set_array(frames[frame])
        ax.set_title(f"{title} (t={frame})")
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(frames), blit=True)
    ani.save(f"animations/{filename}.gif", writer='pillow', fps=2)
    plt.close(fig)

create_animation(mu_frames, "Membership (μ)", "mu_animation", 'YlGnBu')
create_animation(sigma_frames, "Indeterminacy (σ)", "sigma_animation", 'YlOrBr')
create_animation(nu_frames, "Non-membership (ν)", "nu_animation", 'Purples')

print("\n✅ Animated heatmaps saved to animations/*.gif")
print("✅ Neutrosophic region coverage stats saved to neutrosophic_region_areas.csv")


✅ Animated heatmaps saved to animations/*.gif
✅ Neutrosophic region coverage stats saved to neutrosophic_region_areas.csv


In [3]:
fig, axs = plt.subplots(timesteps, 3, figsize=(15, 3 * timesteps))

for t in range(timesteps):
    mu_total = mu_frames[t]
    sigma_total = sigma_frames[t]
    nu_total = nu_frames[t]

    interior = neutrosophic_interior(mu_total)
    boundary = neutrosophic_boundary(sigma_total)
    exterior = neutrosophic_exterior(nu_total)

    axs[t, 0].imshow(interior, extent=(0, 10, 0, 10), origin='lower', cmap='Greens')
    axs[t, 0].set_title(f"Interior (t={t})")

    axs[t, 1].imshow(boundary, extent=(0, 10, 0, 10), origin='lower', cmap='Oranges')
    axs[t, 1].set_title(f"Boundary (t={t})")

    axs[t, 2].imshow(exterior, extent=(0, 10, 0, 10), origin='lower', cmap='Reds')
    axs[t, 2].set_title(f"Exterior (t={t})")

    for j in range(3):
        axs[t, j].set_xticks([])
        axs[t, j].set_yticks([])

plt.tight_layout()
plt.savefig("animations/all_neutrosophic_masks_grid.png")
plt.close()

print("✅ Grid of all neutrosophic masks saved as animations/all_neutrosophic_masks_grid.png")


✅ Grid of all neutrosophic masks saved as animations/all_neutrosophic_masks_grid.png


In [9]:
import copy

def zone_area_fractions(mu, sigma, nu, tau_mu=0.6, tau_sigma=0.4, tau_nu=0.6):
    interior = mu > tau_mu
    boundary = sigma > tau_sigma
    exterior = nu > tau_nu
    total = mu.size
    return {
        "tau_mu": tau_mu,
        "tau_sigma": tau_sigma,
        "tau_nu": tau_nu,
        "Interior_frac": interior.sum() / total,
        "Boundary_frac": boundary.sum() / total,
        "Exterior_frac": exterior.sum() / total,
    }

mu_final = mu_frames[-1]
sigma_final = sigma_frames[-1]
nu_final = nu_frames[-1]


tau_mu_grid    = np.linspace(0.4, 0.8, 9)   # 0.40 … 0.80
tau_sigma_grid = np.linspace(0.2, 0.6, 9)   # 0.20 … 0.60
tau_nu_grid    = np.linspace(0.4, 0.8, 9)   # 0.40 … 0.80

sweep_rows = []
for tmu in tau_mu_grid:
    sweep_rows.append(zone_area_fractions(mu_final, sigma_final, nu_final,
                                          tau_mu=tmu, tau_sigma=0.4, tau_nu=0.6))
for tsig in tau_sigma_grid:
    sweep_rows.append(zone_area_fractions(mu_final, sigma_final, nu_final,
                                          tau_mu=0.6, tau_sigma=tsig, tau_nu=0.6))
for tnu in tau_nu_grid:
    sweep_rows.append(zone_area_fractions(mu_final, sigma_final, nu_final,
                                          tau_mu=0.6, tau_sigma=0.4, tau_nu=tnu))

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("sensitivity_thresholds.csv", index=False)

plt.figure(figsize=(10,4))
plt.subplot(1,3,1)
sel = sweep_df[(sweep_df["tau_sigma"]==0.4)&(sweep_df["tau_nu"]==0.6)]
plt.plot(sel["tau_mu"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_mu"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_mu"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τμ")
plt.xlabel("τμ"); plt.ylabel("Area fraction"); plt.legend(); plt.grid(True)

plt.subplot(1,3,2)
sel = sweep_df[(sweep_df["tau_mu"]==0.6)&(sweep_df["tau_nu"]==0.6)]
plt.plot(sel["tau_sigma"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_sigma"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_sigma"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τσ")
plt.xlabel("τσ"); plt.grid(True)

plt.subplot(1,3,3)
sel = sweep_df[(sweep_df["tau_mu"]==0.6)&(sweep_df["tau_sigma"]==0.4)]
plt.plot(sel["tau_nu"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_nu"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_nu"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τν")
plt.xlabel("τν"); plt.grid(True)
plt.tight_layout()
plt.savefig("sensitivity_thresholds_plots.png", dpi=200)
plt.close()

def layers_from_sensors_list(sensors_list):
    mu_total = np.zeros_like(X)
    sigma_total = np.zeros_like(X)
    nu_total = np.zeros_like(X)
    for s in sensors_list:
        mu_l, sig_l, nu_l, cov = compute_neutrosophic_layers(s, X, Y)
        mu_total += mu_l; sigma_total += sig_l; nu_total += nu_l
    return mu_total, sigma_total, nu_total

def mean_trust_from_layers(mu_l, sig_l, nu_l):
    denom = mu_l + sig_l + nu_l
    trust = np.divide(mu_l, denom, out=np.zeros_like(mu_l), where=denom!=0)
    return float(trust.mean())

base_sensors = [generate_sensor(i) for i in range(num_sensors)]
mu_b, sig_b, nu_b = layers_from_sensors_list(base_sensors)
base = {
    "mean_trust": mean_trust_from_layers(mu_b, sig_b, nu_b),
    **zone_area_fractions(mu_b, sig_b, nu_b, 0.6, 0.4, 0.6)
}

scales = [0.8, 0.9, 1.0, 1.1, 1.2]
rows = []

def apply_scale(sensors_list, key, scale):
    s2 = copy.deepcopy(sensors_list)
    for s in s2:
        if key == "battery":
            s[key] = np.clip(s[key]*scale, 0.0, 1.0)
        else:
            s[key] = max(0.0, s[key]*scale)
    return s2

for param in ["noise", "battery", "drift", "fail_risk"]:
    for sc in scales:
        perturbed = apply_scale(base_sensors, param, sc)
        mu_p, sig_p, nu_p = layers_from_sensors_list(perturbed)
        row = {
            "parameter": param,
            "scale": sc,
            "mean_trust": mean_trust_from_layers(mu_p, sig_p, nu_p),
        }
        row.update(zone_area_fractions(mu_p, sig_p, nu_p, 0.6, 0.4, 0.6))
        rows.append(row)

param_df = pd.DataFrame(rows)
param_df.to_csv("sensitivity_parameters.csv", index=False)

plt.figure(figsize=(10,6))
for param in ["noise", "battery", "drift", "fail_risk"]:
    sub = param_df[param_df["parameter"]==param].sort_values("scale")
    plt.plot(sub["scale"], sub["mean_trust"], marker='o', label=param)
plt.title("Parameter Sensitivity: Mean Trust vs Scale")
plt.xlabel("Scale factor (0.8× to 1.2×)"); plt.ylabel("Mean trust")
plt.grid(True); plt.legend()
plt.tight_layout()
plt.savefig("sensitivity_parameters_mean_trust.png", dpi=200)
plt.close()

plt.figure(figsize=(10,6))
for param in ["noise", "battery", "drift", "fail_risk"]:
    sub = param_df[param_df["parameter"]==param].sort_values("scale")
    plt.plot(sub["scale"], sub["Interior_frac"], marker='o', label=param)
plt.title("Parameter Sensitivity: Interior Fraction vs Scale")
plt.xlabel("Scale factor (0.8× to 1.2×)"); plt.ylabel("Interior area fraction")
plt.grid(True); plt.legend()
plt.tight_layout()
plt.savefig("sensitivity_parameters_interior.png", dpi=200)
plt.close()

print("\n✅ Sensitivity analysis complete.")
print("Saved files:")
print(" - sensitivity_thresholds.csv")
print(" - sensitivity_thresholds_plots.png")
print(" - sensitivity_parameters.csv")
print(" - sensitivity_parameters_mean_trust.png")
print(" - sensitivity_parameters_interior.png")



✅ Sensitivity analysis complete.
Saved files:
 - sensitivity_thresholds.csv
 - sensitivity_thresholds_plots.png
 - sensitivity_parameters.csv
 - sensitivity_parameters_mean_trust.png
 - sensitivity_parameters_interior.png


In [11]:

import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless plotting
import matplotlib.pyplot as plt

OUTDIR = "./"  # or "/mnt/data"
os.makedirs(OUTDIR, exist_ok=True)

grid_size = 100
x_vals = np.linspace(0, 10, grid_size)
y_vals = np.linspace(0, 10, grid_size)
X, Y = np.meshgrid(x_vals, y_vals)

num_sensors = 10
sensor_range = 2.5

RNG_SENS = np.random.default_rng(1003)

BASELINE_JSON = os.path.join(OUTDIR, "sensitivity_baseline_sensors.json")

TAU_MU_NOM  = 0.6
TAU_SIG_NOM = 0.4
TAU_NU_NOM  = 0.6

tau_mu_grid    = np.linspace(0.4, 0.8, 9)  # 0.40 … 0.80
tau_sigma_grid = np.linspace(0.2, 0.6, 9)  # 0.20 … 0.60
tau_nu_grid    = np.linspace(0.4, 0.8, 9)  # 0.40 … 0.80

scales = [0.8, 0.9, 1.0, 1.1, 1.2]


def generate_sensor(i, rng):
    return {
        "id": f"S{i+1}",
        "pos": [float(rng.uniform(1, 9)), float(rng.uniform(1, 9))],
        "range": float(sensor_range),
        "noise": float(rng.uniform(0.05, 0.2)),
        "battery": float(rng.uniform(0.6, 1.0)),
        "drift": float(rng.uniform(0.01, 0.1)),
        "fail_risk": float(rng.uniform(0.01, 0.2)),
    }


def compute_neutrosophic_layers(sensor, X, Y):
    sx, sy = sensor["pos"]
    r = sensor["range"]
    distance = np.sqrt((X - sx) ** 2 + (Y - sy) ** 2)
    coverage = np.clip(1 - distance / r, 0, 1)

    noise = sensor["noise"]
    battery = sensor["battery"]
    drift = sensor["drift"]
    fail_risk = sensor["fail_risk"]

    mu    = coverage * (1 - noise) * battery
    sigma = (noise + drift + (1 - battery)) * coverage
    nu    = ((1 - coverage) * (1 - battery) + fail_risk) * (1 - coverage)

    return mu, sigma, nu, coverage


def layers_from_sensors_list(sensors_list):
    mu_total = np.zeros_like(X)
    sigma_total = np.zeros_like(X)
    nu_total = np.zeros_like(X)
    for s in sensors_list:
        mu_l, sig_l, nu_l, _ = compute_neutrosophic_layers(s, X, Y)
        mu_total   += mu_l
        sigma_total += sig_l
        nu_total   += nu_l
    return mu_total, sigma_total, nu_total


def mean_trust_from_layers(mu_l, sig_l, nu_l):
    denom = mu_l + sig_l + nu_l
    trust = np.divide(mu_l, denom, out=np.zeros_like(mu_l), where=denom != 0)
    return float(trust.mean())


def zone_area_fractions(mu, sigma, nu, tau_mu=TAU_MU_NOM, tau_sigma=TAU_SIG_NOM, tau_nu=TAU_NU_NOM):
    interior = mu > tau_mu
    boundary = sigma > tau_sigma
    exterior = nu > tau_nu
    total = mu.size
    return {
        "tau_mu": tau_mu,
        "tau_sigma": tau_sigma,
        "tau_nu": tau_nu,
        "Interior_frac": float(interior.sum() / total),
        "Boundary_frac": float(boundary.sum() / total),
        "Exterior_frac": float(exterior.sum() / total),
    }


def save_sensors(path, sensors):
    with open(path, "w") as f:
        json.dump(sensors, f)


def load_sensors(path):
    with open(path, "r") as f:
        return json.load(f)


if os.path.exists(BASELINE_JSON):
    base_sensors = load_sensors(BASELINE_JSON)
else:
    base_sensors = [generate_sensor(i, RNG_SENS) for i in range(num_sensors)]
    save_sensors(BASELINE_JSON, base_sensors)

mu_b, sig_b, nu_b = layers_from_sensors_list(base_sensors)


sweep_rows = []
for tmu in tau_mu_grid:
    sweep_rows.append(zone_area_fractions(mu_b, sig_b, nu_b,
                                          tau_mu=tmu, tau_sigma=TAU_SIG_NOM, tau_nu=TAU_NU_NOM))
for tsig in tau_sigma_grid:
    sweep_rows.append(zone_area_fractions(mu_b, sig_b, nu_b,
                                          tau_mu=TAU_MU_NOM, tau_sigma=tsig, tau_nu=TAU_NU_NOM))
for tnu in tau_nu_grid:
    sweep_rows.append(zone_area_fractions(mu_b, sig_b, nu_b,
                                          tau_mu=TAU_MU_NOM, tau_sigma=TAU_SIG_NOM, tau_nu=tnu))

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(os.path.join(OUTDIR, "sensitivity_thresholds.csv"), index=False)

plt.figure(figsize=(10,4))

plt.subplot(1,3,1)
sel = sweep_df[(sweep_df["tau_sigma"]==TAU_SIG_NOM)&(sweep_df["tau_nu"]==TAU_NU_NOM)]
plt.plot(sel["tau_mu"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_mu"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_mu"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τμ"); plt.xlabel("τμ"); plt.ylabel("Area fraction")
plt.grid(True); plt.legend()

plt.subplot(1,3,2)
sel = sweep_df[(sweep_df["tau_mu"]==TAU_MU_NOM)&(sweep_df["tau_nu"]==TAU_NU_NOM)]
plt.plot(sel["tau_sigma"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_sigma"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_sigma"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τσ"); plt.xlabel("τσ"); plt.grid(True); plt.legend()

plt.subplot(1,3,3)
sel = sweep_df[(sweep_df["tau_mu"]==TAU_MU_NOM)&(sweep_df["tau_sigma"]==TAU_SIG_NOM)]
plt.plot(sel["tau_nu"], sel["Interior_frac"], marker='o', label='Interior')
plt.plot(sel["tau_nu"], sel["Boundary_frac"], marker='o', label='Boundary')
plt.plot(sel["tau_nu"], sel["Exterior_frac"], marker='o', label='Exterior')
plt.title("Sweep τν"); plt.xlabel("τν"); plt.grid(True); plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "sensitivity_thresholds_plots.png"), dpi=200)
plt.close()


def apply_scale(sensors_list, key, scale):
    new_list = json.loads(json.dumps(sensors_list))
    for s in new_list:
        if key == "battery":
            s[key] = float(np.clip(s[key]*scale, 0.0, 1.0))
        else:
            s[key] = float(max(0.0, s[key]*scale))
    return new_list

rows = []
for param in ["noise", "battery", "drift", "fail_risk"]:
    for sc in scales:
        perturbed = apply_scale(base_sensors, param, sc)
        mu_p, sig_p, nu_p = layers_from_sensors_list(perturbed)
        record = {
            "parameter": param,
            "scale": sc,
            "mean_trust": mean_trust_from_layers(mu_p, sig_p, nu_p),
        }
        record.update(zone_area_fractions(mu_p, sig_p, nu_p,
                                          TAU_MU_NOM, TAU_SIG_NOM, TAU_NU_NOM))
        rows.append(record)

param_df = pd.DataFrame(rows).sort_values(["parameter", "scale"])
param_df.to_csv(os.path.join(OUTDIR, "sensitivity_parameters.csv"), index=False)

plt.figure(figsize=(10,6))
for param in ["noise", "battery", "drift", "fail_risk"]:
    sub = param_df[param_df["parameter"]==param]
    plt.plot(sub["scale"], sub["mean_trust"], marker='o', label=param)
plt.title("Parameter Sensitivity: Mean Trust vs Scale")
plt.xlabel("Scale factor (0.8× to 1.2×)"); plt.ylabel("Mean trust")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "sensitivity_parameters_mean_trust.png"), dpi=200)
plt.close()

plt.figure(figsize=(10,6))
for param in ["noise", "battery", "drift", "fail_risk"]:
    sub = param_df[param_df["parameter"]==param]
    plt.plot(sub["scale"], sub["Interior_frac"], marker='o', label=param)
plt.title("Parameter Sensitivity: Interior Fraction vs Scale")
plt.xlabel("Scale factor (0.8× to 1.2×)"); plt.ylabel("Interior area fraction")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "sensitivity_parameters_interior.png"), dpi=200)
plt.close()

print("✅ Deterministic sensitivity analysis complete.")
print("Saved:")
print(" - sensitivity_baseline_sensors.json")
print(" - sensitivity_thresholds.csv")
print(" - sensitivity_thresholds_plots.png")
print(" - sensitivity_parameters.csv")
print(" - sensitivity_parameters_mean_trust.png")
print(" - sensitivity_parameters_interior.png")


✅ Deterministic sensitivity analysis complete.
Saved:
 - sensitivity_baseline_sensors.json
 - sensitivity_thresholds.csv
 - sensitivity_thresholds_plots.png
 - sensitivity_parameters.csv
 - sensitivity_parameters_mean_trust.png
 - sensitivity_parameters_interior.png
